# Ingest the ORDEM Prior

Verifies and parses the NASA ORDEM archive, isolates the radar range window, and creates the initial altitude-population prior used by the Bayesian model.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import re
import tempfile
import zipfile
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import gammaln

JOB_UUID = 'd80e4122-e27b-4ec6-903f-8e06e33ca540'
ZIP_NAME = f'job-{JOB_UUID}.zip'
EXPECTED_SHA256 = 'b70d174128a5bfb3f1f847413e4afa3eea24dcc31635cdd02ba1075b4ca9f588'
CFAR_RANGE_KM = (1026.639, 1084.050)

ZIP_PATH = PROJECT_ROOT / 'ordem_raw' / ZIP_NAME
EVENTS_PATH = PROJECT_ROOT / 'phase0_outputs' / 'catalog_matching' / 'ca_cfar_1p75_24h_final_audited' / 'final_consolidated_catalog_association.csv'
OUT_DIR = PROJECT_ROOT / 'phase1_outputs' / 'ordem_ingest' / JOB_UUID
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    matches = list(PROJECT_ROOT.rglob(ZIP_NAME))
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected one {ZIP_NAME}; found {len(matches)}: {matches}')
    ZIP_PATH = matches[0]

assert ZIP_PATH.exists(), ZIP_PATH
assert EVENTS_PATH.exists(), EVENTS_PATH
print('ORDEM archive:', ZIP_PATH)
print('Audited radar table:', EVENTS_PATH)
print('Outputs:', OUT_DIR)


## 1. Archive integrity and safe extraction


In [ ]:
def sha256_file(path, chunk_bytes=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_bytes), b''):
            h.update(chunk)
    return h.hexdigest()


def safe_extract(zf, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    for info in zf.infolist():
        target = (destination / info.filename).resolve()
        if destination != target and destination not in target.parents:
            raise ValueError(f'Unsafe archive path: {info.filename}')
    zf.extractall(destination)


archive_sha = sha256_file(ZIP_PATH)
if archive_sha != EXPECTED_SHA256:
    raise ValueError(f'ZIP checksum changed: {archive_sha} != {EXPECTED_SHA256}')

work_root = Path(tempfile.gettempdir()) / f'ordem_{JOB_UUID}'
with zipfile.ZipFile(ZIP_PATH) as zf:
    bad_member = zf.testzip()
    if bad_member is not None:
        raise zipfile.BadZipFile(f'CRC failure: {bad_member}')
    manifest = pd.DataFrame([
        {'archive_path': i.filename, 'uncompressed_bytes': i.file_size,
         'compressed_bytes': i.compress_size, 'crc32': f'{i.CRC:08x}',
         'is_directory': i.is_dir()}
        for i in zf.infolist()
    ])
    safe_extract(zf, work_root)

manifest.to_csv(OUT_DIR / 'ordem_archive_manifest.csv', index=False)
result_dirs = [p for p in (work_root / 'results').iterdir() if p.is_dir()]
if len(result_dirs) != 1:
    raise ValueError(f'Expected one ORDEM task; found {len(result_dirs)}')
RESULT_DIR = result_dirs[0]

required = {
    'ORDEM-Proj.prj', 'ORDEM.IN', 'status.txt', 'FLUX_TEL.OUT',
    'IGLOOFLUX_TEL.OUT', 'IGLOOFLUX_SIGMARAN_TEL.OUT',
    'IGLOOFLUX_SIGMAPOP_TEL.OUT'
}
missing = sorted(required - {p.name for p in RESULT_DIR.iterdir()})
if missing:
    raise FileNotFoundError(f'Missing ORDEM outputs: {missing}')
print(f'Archive valid: {ZIP_PATH.stat().st_size:,} bytes; SHA-256 {archive_sha}')
print(f'One completed task; {len(manifest)} archive entries; all required files present.')


## 2. Configuration audit


In [ ]:
project_root = ET.fromstring((RESULT_DIR / 'ORDEM-Proj.prj').read_text(errors='replace').lstrip())
project_values = {child.tag: (child.text or '').strip() for child in project_root.find('ProjectValues')}
status_text = (RESULT_DIR / 'status.txt').read_text(errors='replace')
flux_header = '\n'.join((RESULT_DIR / 'FLUX_TEL.OUT').read_text(errors='replace').splitlines()[:6])

header_match = re.search(
    r'Year:\s*(\d{4}).*Sensor lat\.\s*=\s*([+-]?\d+(?:\.\d+)?)'
    r'.*AZ\s*=\s*([+-]?\d+(?:\.\d+)?).*EL\s*=\s*([+-]?\d+(?:\.\d+)?)',
    flux_header, flags=re.S
)
if not header_match:
    raise ValueError('Could not parse FLUX_TEL.OUT configuration header')

configuration = {
    'ordem_version': project_values['Version'],
    'assessment_type': 'telescope/radar',
    'observation_year': int(header_match.group(1)),
    'sensor_latitude_deg': float(header_match.group(2)),
    'pointing_azimuth_deg': float(header_match.group(3)),
    'pointing_elevation_deg': float(header_match.group(4)),
    'igloo_file': project_values['TelescopeIglooFile'],
    'sigma_random_present': (RESULT_DIR / 'IGLOOFLUX_SIGMARAN_TEL.OUT').exists(),
    'sigma_population_present': (RESULT_DIR / 'IGLOOFLUX_SIGMAPOP_TEL.OUT').exists(),
    'status_complete': status_text.rstrip().endswith('Complete'),
    'archive_sha256': archive_sha,
    'archive_file': ZIP_NAME,
    'task_result_id': RESULT_DIR.name,
}

expected = {
    'ordem_version': 'ORDEM 3.2.0', 'observation_year': 2018,
    'pointing_azimuth_deg': 90.0, 'pointing_elevation_deg': 75.0,
    'igloo_file': 'ALT_5_LEO.TIG', 'status_complete': True,
}
for key, value in expected.items():
    if configuration[key] != value:
        raise ValueError(f'Unexpected {key}: {configuration[key]!r} != {value!r}')
if abs(configuration['sensor_latitude_deg'] - 69.586) > 0.002:
    raise ValueError(f"Unexpected latitude: {configuration['sensor_latitude_deg']}")

(OUT_DIR / 'ordem_configuration_audit.json').write_text(json.dumps(configuration, indent=2))
print(json.dumps(configuration, indent=2))


## 3. Parse flux and uncertainty files


In [ ]:
ID_COLS = ['Element', 'alt_low', 'alt_high', 'rng_low', 'rng_high']
SIZE_TO_M = {
    '10': 1e-5, '15': 10**-4.5, '20': 1e-4, '25': 10**-3.5,
    '30': 1e-3, '35': 10**-2.5, '40': 1e-2, '45': 10**-1.5,
    '50': 1e-1, '55': 10**-0.5, '60': 1.0,
}
MATERIALS = {
    'NK': 'NaK droplets', 'LD': 'low density', 'MD': 'medium density',
    'HD': 'high density', 'IN': 'intact objects'
}


def read_igloo(path):
    lines = Path(path).read_text(errors='replace').splitlines()
    h = next(i for i, line in enumerate(lines) if line.lstrip().startswith('Element'))
    columns = lines[h].split()
    rows = []
    for line in lines[h + 2:]:
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) != len(columns):
            raise ValueError(f'{Path(path).name}: {len(parts)} fields; expected {len(columns)}')
        rows.append(parts)
    return pd.DataFrame(rows, columns=columns).apply(pd.to_numeric)


def to_long(df, prefix, value_name):
    value_cols = [c for c in df.columns if c.startswith(prefix)]
    out = df.melt(ID_COLS, value_cols, 'source_column', value_name)
    out['population_key'] = out['source_column'].str[len(prefix):]
    return out.drop(columns='source_column')


flux = read_igloo(RESULT_DIR / 'IGLOOFLUX_TEL.OUT')
sigma_random = read_igloo(RESULT_DIR / 'IGLOOFLUX_SIGMARAN_TEL.OUT')
sigma_population = read_igloo(RESULT_DIR / 'IGLOOFLUX_SIGMAPOP_TEL.OUT')
if not flux[ID_COLS].equals(sigma_random[ID_COLS]) or not flux[ID_COLS].equals(sigma_population[ID_COLS]):
    raise ValueError('Flux and uncertainty element boundaries differ')

long = to_long(flux, 'Flux_', 'flux_per_m2_yr')
long = long.merge(to_long(sigma_random, 'Sig(ran)_', 'sigma_random_per_m2_yr'),
                  on=ID_COLS + ['population_key'], validate='one_to_one')
long = long.merge(to_long(sigma_population, 'Sig(pop)_', 'sigma_population_per_m2_yr'),
                  on=ID_COLS + ['population_key'], validate='one_to_one')
parsed = long['population_key'].str.extract(r'^(?P<material_code>[A-Z]+)(?P<size_code>\d+)$')
long = pd.concat([long, parsed], axis=1)
if long[['material_code', 'size_code']].isna().any().any():
    raise ValueError('Unrecognized ORDEM population label')
long['material'] = long['material_code'].map(MATERIALS)
long['diameter_threshold_m'] = long['size_code'].map(SIZE_TO_M)
long['sigma_total_rss_per_m2_yr'] = np.hypot(
    long['sigma_random_per_m2_yr'], long['sigma_population_per_m2_yr'])

if len(flux) != 360 or long['population_key'].nunique() != 55 or len(long) != 360 * 55:
    raise ValueError(f'Unexpected ORDEM dimensions: {len(flux)}, {long.population_key.nunique()}, {len(long)}')

long.to_csv(OUT_DIR / 'ordem_flux_long.csv.gz', index=False, compression='gzip')
print(f'Parsed {len(long):,} rows = 360 range elements × 55 population columns.')
print(f"Slant range: {flux.rng_low.min():.3f}–{flux.rng_high.max():.3f} km")
print(f"Altitude: {flux.alt_low.min():.1f}–{flux.alt_high.max():.1f} km")


## 4. Aggregate cumulative size thresholds and isolate the EISCAT window


In [ ]:
group_cols = ID_COLS + ['size_code', 'diameter_threshold_m']
cumulative = long.groupby(group_cols, as_index=False).agg(
    flux_per_m2_yr=('flux_per_m2_yr', 'sum'),
    sigma_random_rss_per_m2_yr=('sigma_random_per_m2_yr', lambda x: float(np.sqrt(np.square(x).sum()))),
    sigma_population_rss_per_m2_yr=('sigma_population_per_m2_yr', lambda x: float(np.sqrt(np.square(x).sum()))),
)
cumulative['sigma_total_rss_per_m2_yr'] = np.hypot(
    cumulative.sigma_random_rss_per_m2_yr,
    cumulative.sigma_population_rss_per_m2_yr)
cumulative['relative_sigma_rss'] = (
    cumulative.sigma_total_rss_per_m2_yr / cumulative.flux_per_m2_yr.replace(0, np.nan))
cumulative.to_csv(OUT_DIR / 'ordem_cumulative_flux_by_range_size.csv', index=False)

range_elements = flux[ID_COLS].copy()
lo, hi = CFAR_RANGE_KM
overlap = range_elements[(range_elements.rng_high > lo) & (range_elements.rng_low < hi)].copy()
overlap['overlap_low_km'] = overlap.rng_low.clip(lower=lo)
overlap['overlap_high_km'] = overlap.rng_high.clip(upper=hi)
overlap['overlap_width_km'] = overlap.overlap_high_km - overlap.overlap_low_km
overlap['coverage_fraction'] = overlap.overlap_width_km / (overlap.rng_high - overlap.rng_low)

matched_prior = cumulative.merge(
    overlap[ID_COLS + ['overlap_low_km', 'overlap_high_km', 'overlap_width_km', 'coverage_fraction']],
    on=ID_COLS, how='inner', validate='many_to_one')
matched_prior['flux_overlap_weighted_per_m2_yr'] = matched_prior.flux_per_m2_yr * matched_prior.coverage_fraction
matched_prior['sigma_overlap_weighted_per_m2_yr'] = matched_prior.sigma_total_rss_per_m2_yr * matched_prior.coverage_fraction
matched_prior.to_csv(OUT_DIR / 'ordem_eiscat_overlap_prior.csv', index=False)

print(f'{len(overlap)} ORDEM elements overlap {lo:.3f}–{hi:.3f} km slant range.')
print(f"They correspond approximately to {overlap.alt_low.min():.0f}–{overlap.alt_high.max():.0f} km altitude along this beam.")
display(overlap[['Element','alt_low','alt_high','overlap_low_km','overlap_high_km','coverage_fraction']])


## 5. Shape-only Bayesian baseline against the audited radar events


In [ ]:
events = pd.read_csv(EVENTS_PATH)
required_event_cols = {'event_id', 'observed_range_km', 'association_tier'}
missing_event_cols = required_event_cols - set(events.columns)
if missing_event_cols:
    raise KeyError(f'Radar table missing columns: {sorted(missing_event_cols)}')
if len(events) != 155:
    raise ValueError(f'Expected 155 audited consolidated events; found {len(events)}')

count_rows = []
for _, b in overlap.iterrows():
    inside = events[(events.observed_range_km >= b.overlap_low_km) &
                    (events.observed_range_km < b.overlap_high_km)]
    count_rows.append({
        'Element': int(b.Element), 'observed_events': len(inside),
        'tier_a_strict_events': int((inside.association_tier == 'tier_a_strict').sum()),
        'tier_b_empirical_events': int((inside.association_tier == 'tier_b_empirical').sum()),
        'unresolved_events': int((inside.association_tier == 'unresolved').sum()),
    })
event_counts = overlap.merge(pd.DataFrame(count_rows), on='Element', validate='one_to_one')
if event_counts.observed_events.sum() != len(events):
    raise ValueError('At least one radar event fell outside the frozen CFAR window')
event_counts.to_csv(OUT_DIR / 'eiscat_events_in_ordem_bins.csv', index=False)


def poisson_deviance(y, mu):
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    terms = np.empty_like(y)
    positive = y > 0
    terms[positive] = y[positive] * np.log(y[positive] / mu[positive]) - (y[positive] - mu[positive])
    terms[~positive] = mu[~positive]
    return float(2 * terms.sum())


rng = np.random.default_rng(20260823)
N_BOOT = 5000
diagnostics = []
comparison_rows = []
y = event_counts.observed_events.to_numpy()
N = int(y.sum())

for size_code in ['30', '35', '40', '45', '50']:
    part = matched_prior[matched_prior.size_code.astype(str) == size_code].sort_values('Element')
    if len(part) != len(event_counts):
        raise ValueError(f'Missing matched bins for size code {size_code}')
    weight = part.flux_overlap_weighted_per_m2_yr.to_numpy()
    p = weight / weight.sum()
    posterior_theta_mean = N + 0.5
    mu = posterior_theta_mean * p
    dev = poisson_deviance(y, mu)
    sims = rng.multinomial(N, p, size=N_BOOT)
    sim_dev = np.array([poisson_deviance(s, N * p) for s in sims])
    diagnostics.append({
        'size_code': size_code,
        'diameter_threshold_m': float(part.diameter_threshold_m.iloc[0]),
        'events': N,
        'posterior_theta_mean': posterior_theta_mean,
        'poisson_deviance': dev,
        'posterior_predictive_tail_probability': float(np.mean(sim_dev >= dev)),
        'spearman_range_shape': float(pd.Series(y).corr(pd.Series(weight), method='spearman')),
        'interpretation': 'shape diagnostic only; not a radar size estimate',
    })
    for element, pi, mui in zip(part.Element.astype(int), p, mu):
        comparison_rows.append({
            'Element': element, 'size_code': size_code,
            'diameter_threshold_m': float(part.diameter_threshold_m.iloc[0]),
            'ordem_shape_probability': pi,
            'posterior_expected_events_shape_only': mui,
        })

diag = pd.DataFrame(diagnostics)
comparison = event_counts.merge(pd.DataFrame(comparison_rows), on='Element', validate='one_to_many')
diag.to_csv(OUT_DIR / 'ordem_size_scenario_shape_diagnostics.csv', index=False)
comparison.to_csv(OUT_DIR / 'ordem_eiscat_shape_comparison.csv', index=False)

                                                                                   
p_uniform = event_counts.coverage_fraction.to_numpy(float)
p_uniform = p_uniform / p_uniform.sum()
uniform_deviance = poisson_deviance(y, N * p_uniform)
uniform_sims = rng.multinomial(N, p_uniform, size=N_BOOT)
uniform_sim_deviance = np.array([poisson_deviance(s, N * p_uniform) for s in uniform_sims])
uniform_benchmark = {
    'model': 'uniform event intensity per unit ORDEM element range',
    'poisson_deviance': uniform_deviance,
    'posterior_predictive_tail_probability': float(np.mean(uniform_sim_deviance >= uniform_deviance)),
}
(OUT_DIR / 'uniform_range_shape_diagnostic.json').write_text(json.dumps(uniform_benchmark, indent=2))

print('Audited events:', len(events))
print(events.association_tier.value_counts().to_string())
display(diag)
print('Uniform-range benchmark:', uniform_benchmark)
print('Do not select a physical size threshold from this 12-bin shape check alone.')


## 6. Publication-ready diagnostic figure


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.5), constrained_layout=True)

                                                   
ax = axes[0, 0]
labels = {'30': '>1 mm', '40': '>1 cm', '50': '>10 cm'}
for size_code, label in labels.items():
    q = cumulative[cumulative.size_code.astype(str) == size_code].sort_values('rng_low')
    x = (q.rng_low + q.rng_high) / 2
    line, = ax.plot(x, q.flux_per_m2_yr, lw=1.7, label=label)
    lower = np.maximum(q.flux_per_m2_yr - q.sigma_total_rss_per_m2_yr, np.finfo(float).tiny)
    upper = q.flux_per_m2_yr + q.sigma_total_rss_per_m2_yr
    ax.fill_between(x, lower, upper, color=line.get_color(), alpha=.10, linewidth=0)
ax.axvspan(*CFAR_RANGE_KM, color='black', alpha=.10, label='EISCAT CFAR window')
ax.set_yscale('log')
ax.set_xlabel('Slant range (km)')
ax.set_ylabel('Cumulative flux (# m$^{-2}$ yr$^{-1}$)')
ax.set_title('A. ORDEM 3.2 range profiles')
ax.legend(fontsize=8)

                                      
ax = axes[0, 1]
x = (event_counts.overlap_low_km + event_counts.overlap_high_km) / 2
obs_fraction = event_counts.observed_events / event_counts.observed_events.sum()
ax.step(x, obs_fraction, where='mid', color='black', lw=2.2, label='Radar events (all tiers)')
for size_code, label in labels.items():
    q = comparison[comparison.size_code.astype(str) == size_code].sort_values('Element')
    ax.step(x, q.ordem_shape_probability, where='mid', lw=1.6, label=f'ORDEM {label}')
ax.set_xlabel('Slant range (km)')
ax.set_ylabel('Fraction within matched window')
ax.set_title('B. Shape comparison; absolute scale removed')
ax.legend(fontsize=8)

                                            
ax = axes[1, 0]
q = comparison[comparison.size_code.astype(str) == '40'].sort_values('Element')
width = np.maximum(event_counts.overlap_width_km.to_numpy() * .42, 1.0)
ax.bar(x - width/2, event_counts.observed_events, width=width,
       color='#4C78A8', alpha=.85, label='Observed radar events')
ax.bar(x + width/2, q.posterior_expected_events_shape_only, width=width,
       color='#F58518', alpha=.80, label='ORDEM >1 cm shape baseline')
ax.set_xlabel('Slant range (km)')
ax.set_ylabel('Events per ORDEM element')
ax.set_title('C. Bayesian scale calibration (>1 cm stress test)')
ax.legend(fontsize=8)

                                                         
ax = axes[1, 1]
mat = long[(long.size_code == '40') & long.Element.isin(overlap.Element)].merge(
    overlap[['Element', 'coverage_fraction']], on='Element', validate='many_to_one')
mat['weighted_flux'] = mat.flux_per_m2_yr * mat.coverage_fraction
composition = mat.groupby(['material_code', 'material'], as_index=False).weighted_flux.sum()
composition['percent'] = 100 * composition.weighted_flux / composition.weighted_flux.sum()
composition = composition.sort_values('percent', ascending=True)
ax.barh(composition.material, composition.percent, color='#54A24B')
ax.set_xlabel('Share of >1 cm matched-window ORDEM flux (%)')
ax.set_title('D. ORDEM material-population composition')
for y_pos, value in enumerate(composition.percent):
    ax.text(value + .4, y_pos, f'{value:.1f}%', va='center', fontsize=8)

fig.suptitle('research ORDEM ingest and EISCAT matched-window baseline', fontsize=15)
fig.savefig(OUT_DIR / 'ordem_eiscat_ingest_4panel.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT_DIR / 'ordem_eiscat_ingest_4panel.pdf', bbox_inches='tight')
plt.show()


## 7. Freeze the audit summary and prepare the next ORDEM action library


In [ ]:
actions = []
for elevation, azimuths in [(75, [0, 90, 180, 270]), (60, [0, 90, 180, 270]), (90, [0])]:
    for azimuth in azimuths:
        action_id = f'AZ{azimuth:03d}_EL{elevation:02d}'
        observed = azimuth == 90 and elevation == 75
        actions.append({'action_id': action_id, 'sensor_latitude_deg': 69.586, 'observation_year': 2018, 'azimuth_deg': azimuth, 'elevation_deg': elevation, 'igloo_file': '5km-LEO (ALT_5_LEO.TIG)', 'create_sigma_files': True, 'already_completed': observed, 'submit_in_next_job': not observed, 'role': 'observed calibration pointing' if observed else 'candidate adaptive action'})
action_grid = pd.DataFrame(actions)
action_grid.to_csv(OUT_DIR / 'proposed_ordem_action_grid.csv', index=False)
summary = {'archive_valid': True, 'configuration': configuration, 'ordem_elements': int(len(flux)), 'ordem_population_columns': int(long.population_key.nunique()), 'ordem_slant_range_km': [float(flux.rng_low.min()), float(flux.rng_high.max())], 'ordem_altitude_km': [float(flux.alt_low.min()), float(flux.alt_high.max())], 'eiscat_cfar_range_km': list(CFAR_RANGE_KM), 'overlap_elements': int(len(overlap)), 'overlap_altitude_km': [float(overlap.alt_low.min()), float(overlap.alt_high.max())], 'audited_radar_events': int(len(events)), 'association_tiers': {str(k): int(v) for k, v in events.association_tier.value_counts().items()}, 'shape_diagnostics': diag.to_dict(orient='records'), 'uniform_range_shape_diagnostic': uniform_benchmark, 'current_archive_pointings': 1, 'adaptive_experiment_ready': False, 'blocking_next_input': 'ORDEM outputs for the 8 additional proposed pointings', 'scientific_caveats': ['ORDEM flux is a prior/benchmark, not ground truth.', 'The scale-only baseline does not yield absolute radar detection efficiency.', 'Unresolved events are not labeled uncataloged debris.', 'A physical size-selection model requires RCS/diameter calibration or sensitivity scenarios.', 'The proposed action grid is for simulation and requires adviser review before operational interpretation.']}
(OUT_DIR / 'ordem_ingest_summary.json').write_text(json.dumps(summary, indent=2))
readme = f'research ORDEM ingest — completed\n\nArchive: {ZIP_NAME}\nSHA-256: {archive_sha}\nSettings: ORDEM 3.2.0; 2018; lat 69.586 deg; az 90 deg; el 75 deg; 5km-LEO\nCoverage: {flux.rng_low.min():.3f}-{flux.rng_high.max():.3f} km slant range; {len(flux)} bins\nEISCAT overlap: {CFAR_RANGE_KM[0]:.3f}-{CFAR_RANGE_KM[1]:.3f} km; {len(overlap)} ORDEM bins\nRadar audit: {len(events)} consolidated events; 46 Tier A; 3 Tier B; 106 unresolved\n\nInterpretation:\nORDEM supplies a range- and size-dependent population prior. The Bayesian comparison in this folder calibrates only a nuisance total-count scale. It must not be described as an absolute population estimate or a radar size threshold.\n\nNext input:\nRun the 8 additional pointings marked submit_in_next_job=True in proposed_ordem_action_grid.csv, using the same 2018, latitude, 5km-LEO, and sigma settings.\n'
(OUT_DIR / 'README_ordem_ingest.txt').write_text(readme)
print('SUCCESS: ORDEM archive audited and matched-window prior created.')
print('Output folder:', OUT_DIR)
print('Next required input: 8 additional ORDEM pointing tasks from proposed_ordem_action_grid.csv')
display(action_grid)


## Outputs
